In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import numpy as np
import itertools
import pandas as pd

# =========================================================
# 1. Define stiffness-expression coefficients for the five orientations
#    beta = gamma0 * (c0 + c1*delta1 + c2*delta2)
#         = x1*c0 + x2*c1 + x3*c2
#    where:
#         x1 = gamma0
#         x2 = gamma0 * delta1
#         x3 = gamma0 * delta2
# =========================================================

STIFFNESS_MODELS = {
    "100[010]":   (1.0, -18/5,    -80/7),
    "110[001]":   (1.0, -21/10,   365/14),
    "110[1-10]":  (1.0,  39/10,   155/14),
    "110[1-12]":  (1.0, -1/10,    295/14),
    "111[1-21]":  (1.0,  12/5,   -1280/63),
}


# =========================================================
# 2. Solve for gamma0, delta1, and delta2 from any three orientations
# =========================================================
def solve_from_combo(stiffness_dict, combo, models):
    A = np.array([models[name] for name in combo], dtype=float)
    b = np.array([stiffness_dict[name] for name in combo], dtype=float)

    try:
        x1, x2, x3 = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return None  # mark as an invalid combination

    gamma0 = x1
    delta1 = x2 / x1
    delta2 = x3 / x1
    return gamma0, delta1, delta2


# =========================================================
# 3. Use the fitted parameters to predict stiffness for all orientations
# =========================================================
def predict_stiffness(gamma0, delta1, delta2, models=STIFFNESS_MODELS):
    """
    return dict of predicted stiffness for all orientations
    """
    pred = {}
    for name, (c0, c1, c2) in models.items():
        pred[name] = gamma0 * (c0 + c1 * delta1 + c2 * delta2)
    return pred


# =========================================================
# 4. Compute residuals for all five orientations for a parameter set
# =========================================================
def compute_residuals(gamma0, delta1, delta2, stiffness_dict, models=STIFFNESS_MODELS):
    pred = predict_stiffness(gamma0, delta1, delta2, models=models)

    residuals = {}
    rel_errors = {}
    for name in models:
        residuals[name] = pred[name] - stiffness_dict[name]
        denom = max(abs(stiffness_dict[name]), 1e-12)
        rel_errors[name] = abs(residuals[name]) / denom

    abs_res = np.array(list(residuals.values()))
    rel_res = np.array(list(rel_errors.values()))

    summary = {
        "max_abs_res": np.max(np.abs(abs_res)),
        "l2_abs_res": np.linalg.norm(abs_res),
        "max_rel_res": np.max(rel_res),
        "mean_rel_res": np.mean(rel_res),
    }

    return pred, residuals, rel_errors, summary


# =========================================================
# 5. Loop over all 5-choose-3 combinations
# =========================================================
def solve_all_combinations(stiffness_dict, models=STIFFNESS_MODELS, verbose=True):
    """
    Input:
        stiffness_dict = {
            "100[010]": ...,
            "110[001]": ...,
            "110[1-10]": ...,
            "110[1-12]": ...,
            "111[1-21]": ...,
        }

    Output:
        results_df: parameters and overall residuals for each combination
        details:    more detailed information for each combination
    """
    orientation_names = list(models.keys())
    combos = list(itertools.combinations(orientation_names, 3))

    rows = []
    details = []

    for combo in combos:
        A = np.array([models[name] for name in combo], dtype=float)
        cond_number = np.linalg.cond(A)

        result = solve_from_combo(stiffness_dict, combo, models=models)
        if result is None:
            continue

        gamma0, delta1, delta2 = result

        pred, residuals, rel_errors, summary = compute_residuals(
            gamma0, delta1, delta2, stiffness_dict, models=models
        )

        row = {
            "combo": " + ".join(combo),
            "gamma0": gamma0,
            "delta1": delta1,
            "delta2": delta2,
            "cond(A)": cond_number,
            "max_abs_res": summary["max_abs_res"],
            "l2_abs_res": summary["l2_abs_res"],
            "max_rel_res": summary["max_rel_res"],
            "mean_rel_res": summary["mean_rel_res"],
        }
        rows.append(row)

        details.append({
            "combo": combo,
            "gamma0": gamma0,
            "delta1": delta1,
            "delta2": delta2,
            "predicted": pred,
            "residuals": residuals,
            "relative_errors": rel_errors,
            "cond(A)": cond_number,
        })

    results_df = pd.DataFrame(rows)
    results_df = results_df.sort_values(
        by=["max_abs_res", "max_rel_res", "cond(A)"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    if verbose:
        pd.set_option("display.precision", 10)
        print(results_df)

    return results_df, details


# =========================================================
# 6. Print detailed results for one combination
# =========================================================
def print_combo_detail(detail, stiffness_dict):
    combo = detail["combo"]
    gamma0 = detail["gamma0"]
    delta1 = detail["delta1"]
    delta2 = detail["delta2"]

    print("=" * 80)
    print("Combination:", " + ".join(combo))
    print(f"gamma0 = {gamma0:.12f}")
    print(f"delta1 = {delta1:.12f}")
    print(f"delta2 = {delta2:.12f}")
    print(f"cond(A) = {detail['cond(A)']:.6e}")
    print()

    print("Check against all 5 orientations:")
    for name in STIFFNESS_MODELS:
        pred = detail["predicted"][name]
        true = stiffness_dict[name]
        res = detail["residuals"][name]
        rel = detail["relative_errors"][name]
        print(
            f"{name:12s}  pred={pred: .12f}  true={true: .12f}  "
            f"res={res:+.3e}  rel={rel:.3e}"
        )
    print("=" * 80)


# =========================================================
# 7. Example usage
# =========================================================
if __name__ == "__main__":
    # Replace these with the five stiffness values
    stiffness_input = {
        "100[010]":   85.1E-20,
        "110[001]":   79.7E-20,
        "110[1-10]":  122.8E-20,
        "110[1-12]":  94.4E-20,
        "111[1-21]":  111.2E-20,
    }

    results_df, details = solve_all_combinations(stiffness_input, verbose=True)

    print("\nBest combination based on residuals:")
    print(results_df.iloc[0])

    print("\nDetailed report for the best combination:")
    best_combo_name = results_df.iloc[0]["combo"]
    best_detail = next(d for d in details if " + ".join(d["combo"]) == best_combo_name)
    print_combo_detail(best_detail, stiffness_input)